In [11]:
import nfl_data_py as nfl
import pandas as pd
import json

# Pull 2024 schedule
sched = nfl.import_schedules([2024])
sched = sched[sched['game_type'] == 'REG'][['week', 'home_team', 'away_team']].copy()

# Build lookup: (week, team) -> 'home' or 'away'
location_lookup = {}
for _, row in sched.iterrows():
    location_lookup[(int(row['week']), row['home_team'])] = 'home'
    location_lookup[(int(row['week']), row['away_team'])] = 'away'

print(f"Lookup entries: {len(location_lookup)}")
# Quick test
print(location_lookup.get((1, 'KC'), 'not found'))  # should be home or away

Lookup entries: 544
home


In [12]:
def add_location(records, lookup):
    for r in records:
        key = (int(r['week']), r['recent_team'])
        r['location'] = lookup.get(key, 'home')  # default home if not found
    return records

# Load existing JSON files
for filename in ['qb_data', 'rb_data', 'wr_data', 'te_data']:
    with open(f'{filename}.json') as f:
        data = json.load(f)
    data = add_location(data, location_lookup)
    with open(f'{filename}.json', 'w') as f:
        json.dump(data, f)
    print(f"{filename}: {len(data)} records, sample location: {data[0]['location']}")

qb_data: 660 records, sample location: away
rb_data: 1406 records, sample location: away
wr_data: 2200 records, sample location: home
te_data: 1136 records, sample location: home
